In [13]:
"""
Credit Card Fraud Detection — Real Dataset Pipeline
Columns: trans_date_trans_time, cc_num, merchant, category, amt,
         first, last, gender, street, city, state, zip, lat, long,
         city_pop, job, dob, trans_num, unix_time, merch_lat, merch_long, is_fraud

Usage:
    Place fraudTrain.csv and fraudTest.csv in the same folder as this script.
    Then run:  python fraud_detection.py
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, f1_score
)
from imblearn.over_sampling import SMOTE

print("=" * 65)
print("  CREDIT CARD FRAUD DETECTION — REAL DATASET PIPELINE")
print("=" * 65)

  CREDIT CARD FRAUD DETECTION — REAL DATASET PIPELINE


In [14]:
# ── 1. Load datasets (no splitting — files are already separate) ──────────────
print("\n📂 Loading datasets ...")
train_df = pd.read_csv("fraudTrain.csv")
test_df  = pd.read_csv("fraudTest.csv")
print(f"   fraudTrain.csv  →  {len(train_df):,} rows")
print(f"   fraudTest.csv   →  {len(test_df):,} rows")


📂 Loading datasets ...
   fraudTrain.csv  →  496,140 rows
   fraudTest.csv   →  503,868 rows


In [15]:
# ── 2. Feature Engineering ────────────────────────────────────────────────────
def engineer(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Datetime features
    df['trans_date_trans_time'] = pd.to_datetime(
        df['trans_date_trans_time'], dayfirst=True, errors='coerce')
    df['hour']        = df['trans_date_trans_time'].dt.hour
    df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['month']       = df['trans_date_trans_time'].dt.month
    df['is_night']    = df['hour'].apply(lambda h: 1 if (h < 6 or h >= 22) else 0)

    # Age from date of birth
    df['dob'] = pd.to_datetime(df['dob'], dayfirst=True, errors='coerce')
    ref_date  = pd.Timestamp('2019-01-01')
    df['age'] = ((ref_date - df['dob']).dt.days / 365.25).fillna(40)

    # Haversine distance between cardholder home and merchant
    R   = 6371
    la1 = np.radians(df['lat'].astype(float))
    la2 = np.radians(df['merch_lat'].astype(float))
    lo1 = np.radians(df['long'].astype(float))
    lo2 = np.radians(df['merch_long'].astype(float))
    a   = np.sin((la2 - la1) / 2)**2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2)**2
    df['distance_km'] = 2 * R * np.arcsin(np.sqrt(a))

    # Encode categorical columns
    df['gender_enc']   = (df['gender'] == 'M').astype(int)
    df['category_enc'] = LabelEncoder().fit_transform(df['category'].astype(str))

    # Log-transform skewed numerics
    df['log_amt']      = np.log1p(df['amt'].astype(float))
    df['log_city_pop'] = np.log1p(df['city_pop'].astype(float))

    return df

print("\n🔧 Engineering features ...")
train_df = engineer(train_df)
test_df  = engineer(test_df)

FEATURES = [
    'log_amt', 'hour', 'day_of_week', 'month', 'is_night',
    'age', 'distance_km', 'gender_enc', 'category_enc',
    'log_city_pop', 'zip'
]


🔧 Engineering features ...


In [16]:
# ── 3. Drop rows where target is NaN ─────────────────────────────────────────
print("\n🧹 Dropping rows with NaN in is_fraud target ...")
train_df = train_df.dropna(subset=['is_fraud']).reset_index(drop=True)
test_df  = test_df.dropna(subset=['is_fraud']).reset_index(drop=True)
train_df['is_fraud'] = train_df['is_fraud'].astype(int)
test_df['is_fraud']  = test_df['is_fraud'].astype(int)
print(f"   Train after drop: {len(train_df):,} rows")
print(f"   Test  after drop: {len(test_df):,} rows")

X_train = train_df[FEATURES].copy()
y_train = train_df['is_fraud']
X_test  = test_df[FEATURES].copy()
y_test  = test_df['is_fraud']


🧹 Dropping rows with NaN in is_fraud target ...
   Train after drop: 496,139 rows
   Test  after drop: 503,867 rows


In [17]:
# ── 4. Impute NaNs / Infs in features before scaling ─────────────────────────
print("\n🧹 Checking for NaN / Inf values ...")
X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

nan_train = X_train.isna().sum()
if nan_train.any():
    print("   NaNs found in train — imputing with column medians:")
    for col in nan_train[nan_train > 0].index:
        print(f"     {col}: {nan_train[col]:,} NaN(s)")
else:
    print("   No NaNs in train ✔")

nan_test = X_test.isna().sum()
if nan_test.any():
    print("   NaNs found in test — will be filled with train medians:")
    for col in nan_test[nan_test > 0].index:
        print(f"     {col}: {nan_test[col]:,} NaN(s)")

# Compute medians from TRAIN only to avoid data leakage
train_medians = X_train.median()
X_train.fillna(train_medians, inplace=True)
X_test.fillna(train_medians, inplace=True)
print("   Imputation complete ✔")

print(f"\n✅ Train : {len(X_train):,} rows  |  Fraud = {y_train.sum():,}  ({y_train.mean()*100:.2f}%)")
print(f"✅ Test  : {len(X_test):,} rows  |  Fraud = {y_test.sum():,}  ({y_test.mean()*100:.2f}%)")


🧹 Checking for NaN / Inf values ...
   NaNs found in train — imputing with column medians:
     hour: 297,175 NaN(s)
     day_of_week: 297,175 NaN(s)
     month: 297,175 NaN(s)
   Imputation complete ✔

✅ Train : 496,139 rows  |  Fraud = 3,029  (0.61%)
✅ Test  : 503,867 rows  |  Fraud = 2,125  (0.42%)


In [18]:
# ── 5. Scale features ─────────────────────────────────────────────────────────
print("\n⚙️  Scaling features ...")
scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)   # fit ONLY on train
X_te_sc  = scaler.transform(X_test)        # apply same scale to test



⚙️  Scaling features ...


In [19]:
# ── 6. Handle class imbalance with SMOTE (train only) ─────────────────────────
print("⚖️  Applying SMOTE to training set ...")
sm = SMOTE(random_state=42)
X_tr_res, y_tr_res = sm.fit_resample(X_tr_sc, y_train)
counts = pd.Series(y_tr_res).value_counts()
print(f"   After SMOTE → {counts[0]:,} legitimate  /  {counts[1]:,} fraud")

⚖️  Applying SMOTE to training set ...
   After SMOTE → 493,110 legitimate  /  493,110 fraud


In [20]:
# ── 7. Train models ───────────────────────────────────────────────────────────
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=0.1, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(max_depth=8, min_samples_leaf=10, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, max_depth=12,
                                                   min_samples_leaf=5, n_jobs=-1, random_state=42),
}

results = {}
print("\n🤖 Training on fraudTrain.csv ...\n")
for name, mdl in models.items():
    print(f"   [{name}] fitting ...")
    mdl.fit(X_tr_res, y_tr_res)

    print(f"   [{name}] evaluating on fraudTest.csv ...")
    yp   = mdl.predict(X_te_sc)
    yprb = mdl.predict_proba(X_te_sc)[:, 1]
    cm   = confusion_matrix(y_test, yp)
    tn, fp, fn, tp = cm.ravel()
    auc  = roc_auc_score(y_test, yprb)
    apr  = average_precision_score(y_test, yprb)
    f1   = f1_score(y_test, yp, zero_division=0)

    results[name] = dict(mdl=mdl, yp=yp, yprb=yprb,
                         auc=auc, apr=apr, f1=f1,
                         cm=cm, tn=tn, fp=fp, fn=fn, tp=tp)

    print(f"   ✔  ROC-AUC={auc:.4f}  PR-AUC={apr:.4f}  F1={f1:.4f}  "
          f"TP={tp:,} FP={fp:,} FN={fn:,} TN={tn:,}\n")


🤖 Training on fraudTrain.csv ...

   [Logistic Regression] fitting ...
   [Logistic Regression] evaluating on fraudTest.csv ...
   ✔  ROC-AUC=0.8822  PR-AUC=0.2656  F1=0.0234  TP=1,807 FP=150,631 FN=318 TN=351,111

   [Decision Tree] fitting ...
   [Decision Tree] evaluating on fraudTest.csv ...
   ✔  ROC-AUC=0.9806  PR-AUC=0.5048  F1=0.0985  TP=1,981 FP=36,136 FN=144 TN=465,606

   [Random Forest] fitting ...
   [Random Forest] evaluating on fraudTest.csv ...
   ✔  ROC-AUC=0.9844  PR-AUC=0.6855  F1=0.3630  TP=1,794 FP=5,966 FN=331 TN=495,776



In [21]:
# ── 8. Dashboard ──────────────────────────────────────────────────────────────
print("📊 Building dashboard ...")
CLR  = {'Logistic Regression': '#4C9BE8',
        'Decision Tree':       '#F4845F',
        'Random Forest':       '#4CAF82'}
BG, PAN = '#0d1117', '#161b22'

fig = plt.figure(figsize=(24, 26), facecolor=BG)
fig.suptitle('Credit Card Fraud Detection — Model Dashboard',
             fontsize=22, fontweight='bold', color='white', y=0.985)
gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.52, wspace=0.38)

def mkax(spec):
    ax = fig.add_subplot(spec)
    ax.set_facecolor(PAN)
    ax.tick_params(colors='#aaa', labelsize=9)
    for s in ax.spines.values():
        s.set_edgecolor('#30363d')
    return ax

def ttl(ax, t):
    ax.set_title(t, color='white', fontsize=11, fontweight='bold', pad=8)

ax_dist = mkax(gs[0, 0])
ax_feat = mkax(gs[0, 1])
ax_amt  = mkax(gs[0, 2])
ax_cms  = [mkax(gs[1, i]) for i in range(3)]
ax_roc  = mkax(gs[2, 0:2])
ax_pr   = mkax(gs[2, 2])
ax_auc  = mkax(gs[3, 0])
ax_f1   = mkax(gs[3, 1])
ax_tbl  = mkax(gs[3, 2])

# — Class distribution (train) —
vc = y_train.value_counts().sort_index()
bs = ax_dist.bar(['Legitimate', 'Fraudulent'], vc.values,
                  color=['#2ecc71', '#e74c3c'], width=0.5, edgecolor='none')
for b, v in zip(bs, vc.values):
    ax_dist.text(b.get_x() + b.get_width() / 2, v * 1.01, f'{v:,}',
                 ha='center', color='white', fontsize=10, fontweight='bold')
ttl(ax_dist, 'Class Distribution (Train)')
ax_dist.set_ylabel('Count', color='#aaa', fontsize=9)

# — Feature importances (Random Forest) —
rf_imp = pd.Series(results['Random Forest']['mdl'].feature_importances_,
                   index=FEATURES).sort_values(ascending=True)
ax_feat.barh(rf_imp.index, rf_imp.values,
             color=plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(rf_imp))))
ttl(ax_feat, 'Feature Importances (Random Forest)')
ax_feat.set_xlabel('Importance', color='#aaa', fontsize=9)

# — Log(amount) distribution by class —
for cls, col, lbl in [(0, '#2ecc71', 'Legitimate'), (1, '#e74c3c', 'Fraud')]:
    sub = train_df[train_df['is_fraud'] == cls]['log_amt']
    ax_amt.hist(sub, bins=40, alpha=0.65, color=col, label=lbl, edgecolor='none')
ttl(ax_amt, 'Log(Amount) Distribution by Class')
ax_amt.set_xlabel('log(amt + 1)', color='#aaa', fontsize=9)
ax_amt.legend(facecolor=PAN, edgecolor='#30363d', labelcolor='white', fontsize=8)

# — Confusion matrices —
for ax, (name, res) in zip(ax_cms, results.items()):
    cm_pct = res['cm'].astype(float) / res['cm'].sum(axis=1, keepdims=True) * 100
    sns.heatmap(cm_pct, annot=True, fmt='.1f', ax=ax, cmap='Blues',
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'],
                cbar=False, annot_kws={'size': 11, 'weight': 'bold'}, linewidths=1)
    ax.set_facecolor(PAN)
    ax.tick_params(colors='#aaa', labelsize=9)
    ax.set_title(f'Confusion Matrix\n{name}', color='white',
                 fontsize=10, fontweight='bold', pad=6)
    ax.set_xlabel('Predicted', color='#aaa', fontsize=9)
    ax.set_ylabel('Actual',    color='#aaa', fontsize=9)

# — ROC curves —
ax_roc.plot([0, 1], [0, 1], '--', color='#555', lw=1, label='Random (AUC = 0.50)')
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['yprb'])
    ax_roc.plot(fpr, tpr, lw=2.5, color=CLR[name],
                label=f"{name}  (AUC = {res['auc']:.4f})")
ttl(ax_roc, 'ROC Curves — Evaluated on fraudTest.csv')
ax_roc.set_xlabel('False Positive Rate', color='#aaa', fontsize=10)
ax_roc.set_ylabel('True Positive Rate',  color='#aaa', fontsize=10)
ax_roc.legend(facecolor=PAN, edgecolor='#30363d', labelcolor='white', fontsize=9)

# — Precision-Recall curves —
for name, res in results.items():
    p, r, _ = precision_recall_curve(y_test, res['yprb'])
    ax_pr.plot(r, p, lw=2.5, color=CLR[name],
               label=f"{name.split()[0]}  AP={res['apr']:.3f}")
ttl(ax_pr, 'Precision–Recall Curves')
ax_pr.set_xlabel('Recall',    color='#aaa', fontsize=9)
ax_pr.set_ylabel('Precision', color='#aaa', fontsize=9)
ax_pr.legend(facecolor=PAN, edgecolor='#30363d', labelcolor='white', fontsize=8)

# — ROC-AUC bar —
names = list(results.keys())
aucs  = [results[n]['auc'] for n in names]
b = ax_auc.bar(names, aucs, color=[CLR[n] for n in names], width=0.45)
for bar, v in zip(b, aucs):
    ax_auc.text(bar.get_x() + bar.get_width() / 2, v + 0.002, f'{v:.4f}',
                ha='center', color='white', fontsize=10, fontweight='bold')
ax_auc.set_ylim(0.85, 1.02)
ttl(ax_auc, 'ROC-AUC by Model')
ax_auc.set_ylabel('AUC', color='#aaa', fontsize=9)
ax_auc.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=9)

# — F1-Score bar —
f1s = [results[n]['f1'] for n in names]
b2  = ax_f1.bar(names, f1s, color=[CLR[n] for n in names], width=0.45)
for bar, v in zip(b2, f1s):
    ax_f1.text(bar.get_x() + bar.get_width() / 2, v + 0.005, f'{v:.4f}',
               ha='center', color='white', fontsize=10, fontweight='bold')
ax_f1.set_ylim(0, 1.1)
ttl(ax_f1, 'F1-Score by Model')
ax_f1.set_ylabel('F1', color='#aaa', fontsize=9)
ax_f1.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=9)

# — Summary table —
ax_tbl.axis('off')
ttl(ax_tbl, 'Performance Summary (Test Set)')
rows = [
    [n.replace(' ', '\n'),
     f"{res['auc']:.4f}", f"{res['apr']:.4f}", f"{res['f1']:.4f}",
     f"{res['tp']:,}", f"{res['fp']:,}", f"{res['fn']:,}"]
    for n, res in results.items()
]
tbl = ax_tbl.table(
    cellText=rows,
    colLabels=['Model', 'ROC-AUC', 'PR-AUC', 'F1', 'TP', 'FP', 'FN'],
    cellLoc='center', loc='center', bbox=[0, 0, 1, 1]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8.5)
for (r, c), cell in tbl.get_celld().items():
    cell.set_facecolor('#21262d' if r % 2 == 0 else PAN)
    cell.set_edgecolor('#30363d')
    cell.set_text_props(color='white')

plt.savefig('fraud_detection_dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG)
print("✅  Dashboard saved → fraud_detection_dashboard.png")


📊 Building dashboard ...
✅  Dashboard saved → fraud_detection_dashboard.png


In [22]:
# ── 9. Classification reports ─────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  CLASSIFICATION REPORTS  (evaluated on fraudTest.csv)")
print("=" * 65)
for name, res in results.items():
    print(f"\n▶  {name}")
    print(classification_report(y_test, res['yp'],
                                 target_names=['Legitimate', 'Fraud'],
                                 zero_division=0))

best = max(results, key=lambda n: results[n]['auc'])
print(f"\n🏆  Best model: {best}  (ROC-AUC = {results[best]['auc']:.4f})")
print("=" * 65)


  CLASSIFICATION REPORTS  (evaluated on fraudTest.csv)

▶  Logistic Regression
              precision    recall  f1-score   support

  Legitimate       1.00      0.70      0.82    501742
       Fraud       0.01      0.85      0.02      2125

    accuracy                           0.70    503867
   macro avg       0.51      0.78      0.42    503867
weighted avg       0.99      0.70      0.82    503867


▶  Decision Tree
              precision    recall  f1-score   support

  Legitimate       1.00      0.93      0.96    501742
       Fraud       0.05      0.93      0.10      2125

    accuracy                           0.93    503867
   macro avg       0.53      0.93      0.53    503867
weighted avg       1.00      0.93      0.96    503867


▶  Random Forest
              precision    recall  f1-score   support

  Legitimate       1.00      0.99      0.99    501742
       Fraud       0.23      0.84      0.36      2125

    accuracy                           0.99    503867
   macro avg